# Step 1 — Intent classification (IndoBERT-base-p2)

Multi-label classification: each utterance can express more than one intent (e.g. "tambah es teh, terus batalin ayam goreng" = order_add + cancel). Labels come from `data/normalized/augmented_ordering_dataset_normalized.jsonl`'s `intents` field (list of intent strings per row, 1+ per row). Fine-tunes `indobenchmark/indobert-base-p1` (`AutoModelForSequenceClassification`, `problem_type="multi_label_classification"`) on `text_normalized`. `p1` chosen over `p2` for smaller/faster inference, since intent classification runs on every utterance in the live pipeline.

In [1]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p1"
SRC = Path("../data/normalized/augmented_ordering_dataset_normalized.jsonl")
OUT_DIR = Path("../data/intent")

## Step 2 — Load and inspect labels

Synthetic disfluency augmentation (`3.1`'s 60 added `FS`/`RP`/`RM` examples) landed here too, all tagged `order_create` — mild skew (260 vs 200 for other classes) but not severe enough to need weighted loss.

In [2]:
rows = [json.loads(line) for line in SRC.open(encoding="utf-8")]
print(f"{len(rows)} records loaded")

intent_counts = Counter(intent for r in rows for intent in r["intents"])
for intent, count in intent_counts.most_common():
    print(f"  {intent:22s}: {count}")

n_multi = sum(1 for r in rows if len(r["intents"]) > 1)
print(f"\nmulti-intent rows: {n_multi} / {len(rows)}")


1943 records loaded
  menu_inquiry          : 262
  order_status          : 224
  order_remove_item     : 198
  order_modify_quantity : 197
  complaint             : 197
  repeat_request        : 196
  ask_recommendation    : 194
  deny                  : 192
  ask_price             : 191
  cancel                : 188
  confirm               : 183
  order_swap            : 176
  order_add             : 155
  other                 : 142

multi-intent rows: 628 / 1943


In [3]:
label_list = sorted(intent_counts.keys())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'ask_price': 0, 'ask_recommendation': 1, 'cancel': 2, 'complaint': 3, 'confirm': 4, 'deny': 5, 'menu_inquiry': 6, 'order_add': 7, 'order_modify_quantity': 8, 'order_remove_item': 9, 'order_status': 10, 'order_swap': 11, 'other': 12, 'repeat_request': 13}


## Step 3 — Tokenize

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(text: str) -> dict:
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    return {"input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"]}


def multi_hot(intents: list[str]) -> list[float]:
    vec = [0.0] * len(label_list)
    for intent in intents:
        vec[label2id[intent]] = 1.0
    return vec


encoded_rows = []
for r in rows:
    enc = encode(r["text_normalized"])
    encoded_rows.append({
        "id": r["id"],
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": multi_hot(r["intents"]),
    })

print(f"{len(encoded_rows)} records encoded")


1943 records encoded


## Step 4 — Stratified 80/10/10 split

Stratified by each row's primary (first-listed) `intents[0]` to keep class balance across splits, same convention as `scripts/split_disfluency_repair_dataset.py`.

In [5]:
from sklearn.model_selection import train_test_split

VAL_FRAC = 0.1
TEST_FRAC = 0.1

# stratify by the primary (first-listed) intent — sklearn needs a single label
# per row for stratification; multi-hot labels themselves are unaffected.
labels_strat = [r["intents"][0] for r in rows]
train, rest = train_test_split(
    encoded_rows, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_ids = {r["id"] for r in rest}
rest_strat = [r["intents"][0] for r in rows if r["id"] in rest_ids]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_strat, random_state=SEED
)

print(f"train={len(train)} val={len(val)} test={len(test)}")


train=1554 val=194 test=195


In [6]:
def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/intent


## Step 5 — Load splits as `datasets.Dataset`

In [7]:
from datasets import Dataset


def load_split(jsonl_path: Path) -> Dataset:
    records = [json.loads(line) for line in jsonl_path.open(encoding="utf-8")]
    return Dataset.from_list([
        {"input_ids": r["input_ids"], "attention_mask": r["attention_mask"], "labels": r["labels"]}
        for r in records
    ])


train_dataset = load_split(OUT_DIR / "train.jsonl")
val_dataset = load_split(OUT_DIR / "val.jsonl")
test_dataset = load_split(OUT_DIR / "test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1554
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 194
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 195
})


## Step 6 — Model architecture

Full fine-tune of `indobenchmark/indobert-base-p1` via `AutoModelForSequenceClassification`.

In [8]:
from transformers import AutoModelForSequenceClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")


[transformers] You passed `num_labels=14` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device          : mps
total params    : 124,452,110
trainable params: 124,452,110  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [9]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], dtype=torch.float, device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)

print(f"logits shape: {tuple(outputs.logits.shape)}  (expect (4, {len(label_list)}))")
assert outputs.logits.shape == (4, len(label_list))
print(f"sample loss (BCEWithLogitsLoss): {outputs.loss.item():.4f}")

model.train()
print("Step 6 sanity checks passed.")


logits shape: (4, 14)  (expect (4, 14))
sample loss (BCEWithLogitsLoss): 0.6981
Step 6 sanity checks passed.


## Step 7 — Training

Plain `Trainer` (unweighted) — class imbalance here is mild (1.3x at most), unlike the disfluency/NER stages, so the inverse-frequency weighting used there isn't warranted.

In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

THRESHOLD = 0.5


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)
    return {
        "accuracy": accuracy_score(labels.ravel(), preds.ravel()),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [11]:
from ordo_ai.tracking import init_experiment

init_experiment("intent-classification")

In [12]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/indobert-intent",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_strategy="epoch",
    report_to=["mlflow"],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
import mlflow

run = mlflow.start_run()
train_result = trainer.train()
train_result.metrics

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.396485,0.275554,0.907585,0.095609,0.214286,0.071882
2,0.217922,0.173513,0.957658,0.655734,0.852041,0.572212
3,0.149846,0.132681,0.973122,0.818981,0.981647,0.753454
4,0.118503,0.112907,0.978277,0.862544,0.971890,0.810683
5,0.104003,0.105705,0.981959,0.884291,0.984336,0.838729


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 217.6259,
 'train_samples_per_second': 35.703,
 'train_steps_per_second': 2.252,
 'total_flos': 255574145917440.0,
 'train_loss': 0.19735173595194913,
 'epoch': 5.0}

### Save the fine-tuned model

In [14]:
final_model_path = "../models/indobert-intent-final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")

mlflow.log_artifact(final_model_path, artifact_path="model")
mlflow.end_run()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model -> ../models/indobert-intent-final
🏃 View run vaunted-ape-44 at: http://localhost:5001/#/experiments/3/runs/5429b31f58b2400d84fc5c029604b83b
🧪 View experiment at: http://localhost:5001/#/experiments/3


## Step 8 — Evaluation

Held-out **test** split, per-intent precision/recall/F1 via `classification_report` — not just overall accuracy, since `chitchat_oos`/`repeat_request` (semantically closer to other intents) are the likeliest weak spots.

In [15]:
test_predictions = trainer.predict(test_dataset)
test_logits, test_labels = test_predictions.predictions, test_predictions.label_ids
test_probs = 1 / (1 + np.exp(-test_logits))
test_preds = (test_probs >= THRESHOLD).astype(int)
test_labels = test_labels.astype(int)

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


overall test metrics:
  test_loss                : 0.10457388311624527
  test_accuracy            : 0.9857142857142858
  test_f1_macro            : 0.9099707067288455
  test_precision_macro     : 1.0
  test_recall_macro        : 0.855176199770526
  test_runtime             : 1.2644
  test_samples_per_second  : 154.221
  test_steps_per_second    : 10.281


In [16]:
from sklearn.metrics import classification_report

report = classification_report(
    test_labels, test_preds, target_names=label_list, digits=3, zero_division=0
)
print(report)


                       precision    recall  f1-score   support

            ask_price      1.000     1.000     1.000        19
   ask_recommendation      1.000     0.882     0.938        17
               cancel      1.000     0.826     0.905        23
            complaint      1.000     1.000     1.000        23
              confirm      1.000     0.316     0.480        19
                 deny      1.000     0.706     0.828        17
         menu_inquiry      1.000     0.957     0.978        23
            order_add      1.000     0.812     0.897        16
order_modify_quantity      1.000     1.000     1.000        18
    order_remove_item      1.000     0.840     0.913        25
         order_status      1.000     0.950     0.974        20
           order_swap      1.000     1.000     1.000        21
                other      1.000     0.833     0.909        18
       repeat_request      1.000     0.850     0.919        20

            micro avg      1.000     0.860     0.925 

### Misclassified examples

In [17]:
test_records = [json.loads(line) for line in (OUT_DIR / "test.jsonl").open(encoding="utf-8")]
rows_by_id = {r["id"]: r for r in rows}

n_mismatch = 0
shown = 0
for rec, true_vec, pred_vec in zip(test_records, test_labels, test_preds):
    if (true_vec == pred_vec).all():
        continue
    n_mismatch += 1
    if shown < 15:
        text = rows_by_id[rec["id"]]["text_normalized"]
        true_labels = [label_list[i] for i, v in enumerate(true_vec) if v]
        pred_labels = [label_list[i] for i, v in enumerate(pred_vec) if v]
        print(f"  text={text!r}  true={true_labels}  pred={pred_labels}")
        shown += 1

print(f"\nMismatched records: {n_mismatch} / {len(test_records)}")


  text='eh yang tadi pesen mie ayam bakso dan batagor cancel deh terus juga boleh minta tisu ya'  true=['cancel', 'other']  pred=['cancel']
  text='eh tunggu dulu itu lele penyet yang tadi itu sebenarnya gak hmm saya mau tambah kwetiau goreng terus juga hmm ya tambah satu bebek goreng lagi deh hmm tapi kali ini tanpa tambah ayam ya'  true=['deny', 'order_add']  pred=['order_add']
  text='eh boleh ya tanya ada lele goreng sama tahu isinya bentar saya cek oke deh kalau bisa tambahin es teler juga tapi nanti kalau datangin pesenannya tolong ulangi ya mungkin saya mau batalkan ifumienya nih'  true=['cancel', 'menu_inquiry', 'repeat_request']  pred=['cancel', 'menu_inquiry']
  text='emm sebentar ya rekomendasinya ada yang enak dari nasi kuning atau bubur kacang hijau terus juga eh tambah deh saya mau nasi kuning ini ya tambah selada juga ya saya pilih dulu ya minuman es coklatnya'  true=['ask_recommendation', 'confirm']  pred=['ask_recommendation']
  text='oh iya juga saya mau pesen kwetiau

## Step 9 — Inference

Loads the saved model fresh from disk to confirm standalone reload. `predict_intent(text)` tokenizes raw text and returns the predicted intent plus per-class probabilities.

In [18]:
from transformers import AutoModelForSequenceClassification as _AMFSC, AutoTokenizer as _AT
import torch.nn.functional as F

INFERENCE_MODEL_PATH = "../models/indobert-intent-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = _AMFSC.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded model from ../models/indobert-intent-final
id2label: {0: 'ask_price', 1: 'ask_recommendation', 2: 'cancel', 3: 'complaint', 4: 'confirm', 5: 'deny', 6: 'menu_inquiry', 7: 'order_add', 8: 'order_modify_quantity', 9: 'order_remove_item', 10: 'order_status', 11: 'order_swap', 12: 'other', 13: 'repeat_request'}


In [19]:
def predict_intent(text: str) -> dict:
    encoding = inference_tokenizer(
        text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = inference_model(**encoding).logits[0]
    probs = torch.sigmoid(logits)

    all_probs = {id2label[i]: probs[i].item() for i in range(len(probs))}
    active = {label: p for label, p in all_probs.items() if p >= THRESHOLD}
    if not active:
        top_label = max(all_probs, key=all_probs.get)
        active = {top_label: all_probs[top_label]}

    return {
        "intents": list(active.keys()),
        "confidences": active,
        "probs": all_probs,
    }


### Try it on fresh, unseen sentences

In [20]:
demo_sentences = [
    "eh gue mau pesen nasi goreng spesial dua porsi",
    "tambahin es teh manis satu lagi ya",
    "batalin ayam bakar yang tadi",
    "ganti jadi tiga porsi aja",
    "ada menu apa aja sih disini",
    "iya bener itu pesanannya",
    "gak usah jadi deh",
    "tolong ulang lagi pesanannya",
    "boleh minta tisu",
    "mau tanya pesanan saya apa saja",
    "tambahin es teh satu, terus batalin ayam bakarnya ya",
]

for text in demo_sentences:
    result = predict_intent(text)
    print(f"input     : {text}")
    print(f"intents   : {result['intents']}  (confidences={ {k: round(v, 3) for k, v in result['confidences'].items()} })")
    print()


input     : eh gue mau pesen nasi goreng spesial dua porsi
intents   : ['order_modify_quantity']  (confidences={'order_modify_quantity': 0.734})

input     : tambahin es teh manis satu lagi ya
intents   : ['order_add']  (confidences={'order_add': 0.864})

input     : batalin ayam bakar yang tadi
intents   : ['cancel']  (confidences={'cancel': 0.637})

input     : ganti jadi tiga porsi aja
intents   : ['order_modify_quantity']  (confidences={'order_modify_quantity': 0.772})

input     : ada menu apa aja sih disini
intents   : ['menu_inquiry']  (confidences={'menu_inquiry': 0.868})

input     : iya bener itu pesanannya
intents   : ['confirm']  (confidences={'confirm': 0.137})

input     : gak usah jadi deh
intents   : ['order_remove_item']  (confidences={'order_remove_item': 0.284})

input     : tolong ulang lagi pesanannya
intents   : ['other']  (confidences={'other': 0.287})

input     : boleh minta tisu
intents   : ['other']  (confidences={'other': 0.887})

input     : mau tanya pesan